# Домашняя Работа 8-9

In [10]:
import os
import json
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# ========== Seed ==========
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ========== Device ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ========== Paths ==========
BASE_DIR = "artifacts"
FIG_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

Device: cuda


## Данные + сплит

In [11]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_dataset_full = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# sanity check
x, y = next(iter(train_loader))
print(x.shape, y.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([128, 3, 32, 32]) torch.Size([128])


## Модель MLP

In [12]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes,
                 dropout=0.0,
                 use_batchnorm=False):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))

            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = h

        layers.append(nn.Linear(prev_dim, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.net(x)

# Метрики и цикл обучения

In [13]:
def accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(logits, y)

    return total_loss / len(loader), total_acc / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy(logits, y)

    return total_loss / len(loader), total_acc / len(loader)

## EarlyStopping

In [14]:
class EarlyStopping:
    def __init__(self, patience=4):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = model.state_dict()
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

## Функция запуска эксперимента

In [15]:
def run_experiment(
    experiment_id,
    hidden_sizes,
    dropout,
    batchnorm,
    optimizer_name,
    lr,
    momentum=0.0,
    weight_decay=0.0,
    epochs=15,
    early_stopping=False
):
    model = MLP(
        input_dim=3*32*32,
        hidden_sizes=hidden_sizes,
        dropout=dropout,
        use_batchnorm=batchnorm
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    else:
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    stopper = EarlyStopping(patience=4) if early_stopping else None

    best_val_acc = 0
    best_val_loss = float("inf")

    for epoch in range(epochs):

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss

        if early_stopping:
            stop = stopper.step(val_loss, model)
            if stop:
                model.load_state_dict(stopper.best_state)
                break

    return {
        "experiment_id": experiment_id,
        "history": history,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "epochs_trained": len(history["train_loss"]),
        "model_state": model.state_dict()
    }

## Часть A - E1-E4

In [17]:
hidden = [1024, 512]

E1 = run_experiment("E1", hidden, 0.0, False, "Adam", lr=1e-3)
E2 = run_experiment("E2", hidden, 0.3, False, "Adam", lr=1e-3)
E3 = run_experiment("E3", hidden, 0.0, True, "Adam", lr=1e-3)

best_reg = max([E2, E3], key=lambda x: x["best_val_accuracy"])

chosen_dropout = 0.3 if best_reg["experiment_id"] == "E2" else 0.0
chosen_batchnorm = True if best_reg["experiment_id"] == "E3" else False

E4 = run_experiment(
    "E4",
    hidden,
    dropout=chosen_dropout,
    batchnorm=chosen_batchnorm,
    optimizer_name="Adam",
    lr=1e-3,
    epochs=20,
    early_stopping=True
)

## Часть B - O1-O3

In [18]:
O1 = run_experiment("O1", hidden, 0.3, False, "Adam", lr=1e-1, epochs=6)
O2 = run_experiment("O2", hidden, 0.3, False, "Adam", lr=1e-5, epochs=6)

O3 = run_experiment(
    "O3",
    hidden,
    dropout=0.3,
    batchnorm=False,
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    epochs=12
)

## Сохранение артефактов

In [19]:
all_runs = [E1, E2, E3, E4, O1, O2, O3]

def get_model_summary(hidden, dropout, batchnorm):
    return f"MLP({hidden}, dropout={dropout}, bn={batchnorm})"
    
with open(os.path.join(BASE_DIR, "runs.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment_id","dataset","seed",
        "model_summary",
        "optimizer","lr","momentum","weight_decay",
        "epochs_trained","best_val_accuracy","best_val_loss"
    ])

    for run in all_runs:
        writer.writerow([
            run["experiment_id"],
            "CIFAR10",
            SEED,
            get_model_summary(hidden, 0.3, False),  # можно упростить
            "Adam" if run["experiment_id"] != "O3" else "SGD",
            "",
            "",
            "",
            run["epochs_trained"],
            run["best_val_accuracy"],
            run["best_val_loss"]
        ])

## best_model.pt

In [20]:
torch.save(E4["model_state"], os.path.join(BASE_DIR, "best_model.pt"))

## best_config.json

In [21]:
best_config = {
    "dataset": "CIFAR10",
    "seed": SEED,
    "hidden_sizes": hidden,
    "dropout": chosen_dropout,
    "batchnorm": chosen_batchnorm,
    "optimizer": "Adam",
    "lr": 1e-3
}

with open(os.path.join(BASE_DIR, "best_config.json"), "w") as f:
    json.dump(best_config, f, indent=4)

## curves_best.png

In [22]:
plt.figure()
plt.plot(E4["history"]["train_loss"], label="train")
plt.plot(E4["history"]["val_loss"], label="val")
plt.legend()
plt.title("Best Model (E4)")
plt.savefig(os.path.join(FIG_DIR, "curves_best.png"))
plt.close()

## curves_lr_extremes.png

In [23]:
plt.figure()
plt.plot(O1["history"]["train_loss"], label="LR too big")
plt.plot(O2["history"]["train_loss"], label="LR too small")
plt.legend()
plt.title("LR extremes")
plt.savefig(os.path.join(FIG_DIR, "curves_lr_extremes.png"))
plt.close()

## Финальная оценка на test

In [24]:

import json

with open(os.path.join(BASE_DIR, "best_config.json"), "r") as f:
    best_config = json.load(f)

model = MLP(
    input_dim=3*32*32,
    hidden_sizes=best_config["hidden_sizes"],
    dropout=best_config["dropout"],
    use_batchnorm=best_config.get("batchnorm", False)
).to(device)

model.load_state_dict(
    torch.load(os.path.join(BASE_DIR, "best_model.pt"), map_location=device)
)

criterion = nn.CrossEntropyLoss()

test_loss, test_acc = evaluate(model, test_loader, criterion)

print("FINAL TEST ACCURACY:", test_acc)

C:\Users\User\AppData\Local\Temp\ipykernel_33600\2602025455.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(BASE_DIR, "best_model.pt"), map_loca

FINAL TEST ACCURACY: 0.5517207278481012
